In [0]:
import re
from bs4 import BeautifulSoup


def extract_bronze_detail(html_content):
    ticker_map = {
            "msft": "Microsoft Corporation",
            "aapl": "Apple Inc.",
            "goog": "Alphabet Inc.",
            "amzn": "Amazon.com Inc.",
            "nvda": "NVIDIA Inc."
        }
    soup = BeautifulSoup(html_content, "html.parser")

    # 1. Extract CIK
    cik_tag = soup.find("ix:nonnumeric", attrs={"name": "dei:EntityCentralIndexKey"})
    cik = cik_tag.text.strip() if cik_tag else None

    # Fallback CIK check via xbrli:identifier
    if not cik:
        identifier_tag = soup.find("xbrli:identifier")
        if identifier_tag:
            cik = identifier_tag.text.strip()

    # 2. Extract Filing Date (from HTML comments)
    date=None
    date_tag= soup.find("ix:nonnumeric",attrs={"name":"dei:CurrentFiscalYearEndDate"})
    year_tag=soup.find("ix:nonnumeric",attrs={"name":"dei:DocumentFiscalYearFocus"})
    
    if date_tag and year_tag:
        date=",".join([date_tag.get_text().strip(),year_tag.get_text().strip()])

    # 3. Extract Company Name
    # Checked via the schema reference (e.g., msft-20260630.xsd)
    schema_ref=soup.find("link:schemaref")
    company_name = None
    if schema_ref and "xlink:href" in schema_ref.attrs:
       
        ticker = schema_ref["xlink:href"].split("-")[0]
        
        company_name = ticker_map.get(
            ticker.lower(), ticker.upper()
        )  # Defaults to ticker symbol if name is not mapped

    # 4. Extract Accession Number (Present in SEC Header / Filename if loaded from SEC raw file)
    accession_match = re.search(
        r"ACCESSION NUMBER:\s*([\d-]+)", html_content, re.IGNORECASE
    )
    accession_number = accession_match.group(1) if accession_match else "N/A"

    return {
        "company_name": company_name,
        "cik": cik,
        "filling_date": date if date else "N/A",
        "accession_number": accession_number,
        "filling_type":"10K"
    }




In [0]:
#for now we will just be handling html files
def ingest_html(path):
    print("processing file ",path)
    with open(path,encoding="utf-8") as f:
        html=f.read()
        results = extract_bronze_detail(html)
        results["raw_data"]=html
        results["file_name"]=path.split("/")[-1]
        results["doc_id"]=results["cik"]+'_'+results["filling_date"]+"_"+results["filling_type"]
        return results


In [0]:
#schema for bronze
from pyspark.sql.types import *

bronze_schema = StructType([
    StructField("company_name", StringType(), True),
    StructField("cik", StringType(), True),
    StructField("filling_date", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("filling_type", StringType(), True),
    StructField("raw_data", StringType(), True),
    StructField("doc_id",StringType(),True),
    StructField("checksum",StringType(),True),
    StructField("ingestion_timestamp",StringType(),True),
    StructField("last_modified",StringType(),True),
    StructField("file_name",StringType(),True)
])


In [0]:
from pyspark.sql import SparkSession
import time
import hashlib
#constants 
path="/Volumes/workspace/rag/raw_documents"
files=dbutils.fs.ls("/Volumes/workspace/rag/raw_documents")
bronze_table="workspace.rag.bronze_delta"

#set up spark session
spark = SparkSession.builder.appName("Bronze_handling").getOrCreate()


#implementing idempotency and incremental load using doc_id and checksum
if spark.catalog.tableExists(bronze_table):
    df= spark.table(bronze_table)
    max_modified=df.select("last_modified").agg({"last_modified":"max"}).collect()[0][0]
    new_data=[]
    for file in files:
        if file.modificationTime>max_modified:
            data = ingest_html(path+'/'+file.name)
            data["checksum"]=hashlib.sha256(data["raw_data"].encode("UTF-8")).hexdigest()
            data["last_modified"]=file.modificationTime
            data["ingestion_timestamp"]=time.time_ns() // 1_000_000
            if df.filter(df.doc_id==data["doc_id"]).count()>0:
                same_files=df.filter(df.doc_id==data["doc_id"])
                if same_files.filter(same_files.checksum==data["checksum"]).count()>0:
                    print("skipping file ",file.name," as it already exists")
                    continue
                else:
                    print("adding the new version for the file")
                    new_data+=[data]
            else:
                new_data+=[data]
    if len(new_data)>0:
        spark.createDataFrame(new_data,schema=bronze_schema).write.mode("append").saveAsTable(bronze_table)
else:
    bronze_data=[]
    for file in files:
    
        if file.name.endswith(".htm") or file.name.endswith("html"):
            data = ingest_html(path+'/'+file.name)
            data["last_modified"]=file.modificationTime
            data["ingestion_timestamp"]=time.time_ns() // 1_000_000
            data["checksum"]=hashlib.sha256(data["raw_data"].encode("UTF-8")).hexdigest()
            bronze_data+=[data]
            # spark.createDataFrame([data]).write.mode("append").saveAsTable("bronze")
        else:
            print("ignoring file with extension ",file.name.split(".")[-1], " named ",file.name)
    df=spark.createDataFrame(bronze_data,schema=bronze_schema)
    df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)


  



    